In [4]:
# ============================================================
# CELL 1 - Environment Setup
# Install all required dependencies for MMSegmentation + SegFormer
# ============================================================

# Install PyTorch with CUDA 12.1 support
!pip install torch==2.4.1 torchvision==0.19.1 --index-url https://download.pytorch.org/whl/cu121 -q

# Install OpenMMLab stack
!pip install mmengine -q
!pip install mmcv==2.2.0 -f https://download.openmmlab.com/mmcv/dist/cu121/torch2.4/index.html -q
!pip install mmsegmentation -q

# Fix mmcv version check (mmseg 1.2.2 hardcodes max version as 2.2.0 exclusive)
!sed -i 's/MMCV_MAX = .2.2.0./MMCV_MAX = "2.3.0"/' \
    /usr/local/lib/python3.12/dist-packages/mmseg/__init__.py

print("Installation complete, version check fixed!")

Installation complete, version check fixed!


In [5]:
# ============================================================
# CELL 2 - Environment Verification
# Verify all packages are correctly installed and GPU is available
# ============================================================

import torch, mmcv, mmengine, mmseg

print('torch:', torch.__version__)
print('mmcv:', mmcv.__version__)
print('mmseg:', mmseg.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

torch: 2.4.1+cu121
mmcv: 2.2.0
mmseg: 1.2.2
CUDA available: True
GPU: Tesla T4


In [7]:
# ============================================================
# CELL 3 - Mount Google Drive and Extract Dataset
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

# Extract dataset from Drive to Colab local storage (faster I/O during training)
!unzip -q /content/drive/MyDrive/Dataset/processed.zip -d /content/

# Verify dataset structure and image counts
print("=== Dataset Verification ===")
!echo "Train images:" && ls /content/processed/train/images/ | wc -l
!echo "Train masks: " && ls /content/processed/train/masks/  | wc -l
!echo "Val images:  " && ls /content/processed/val/images/   | wc -l
!echo "Val masks:   " && ls /content/processed/val/masks/    | wc -l
!echo "Test images: " && ls /content/processed/test/images/  | wc -l
!echo "Test masks:  " && ls /content/processed/test/masks/   | wc -l

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
=== Dataset Verification ===
Train images:
672
Train masks: 
672
Val images:  
149
Val masks:   
149
Test images: 
146
Test masks:  
146


In [8]:
# ============================================================
# CELL 4 - Clone Repository and Download Pretrained Weights
# ============================================================

import os

# Clone the project repository
!git clone https://github.com/ilMassy/advertising-panel-segmentation /content/repo
%cd /content/repo

# Create checkpoints directory (models/ and results/ already exist in the repo)
os.makedirs('checkpoints', exist_ok=True)

# Download MIT-B0 pretrained weights (for baseline experiment)
!wget -q --show-progress \
    https://download.openmmlab.com/mmsegmentation/v0.5/pretrain/segformer/mit_b0_20220624-7e0fe6dd.pth \
    -O checkpoints/mit_b0_imagenet.pth

# Download MIT-B1 pretrained weights (for main experiment)
!wget -q --show-progress \
    https://download.openmmlab.com/mmsegmentation/v0.5/pretrain/segformer/mit_b1_20220624-02e5a6a1.pth \
    -O checkpoints/mit_b1_imagenet.pth

# Verify downloads
!echo "Checkpoint sizes:"
!ls -lh checkpoints/

Cloning into '/content/repo'...
remote: Enumerating objects: 80, done.
remote: Counting objects: 100% (80/80), done.
remote: Compressing objects: 100% (60/60), done.
remote: Total 80 (delta 29), reused 57 (delta 14), pack-reused 0 (from 0)
Receiving objects: 100% (80/80), 20.57 KiB | 752.00 KiB/s, done.
Resolving deltas: 100% (29/29), done.
/content/repo
checkpoints/mit_b0_ 100%[===================>]  12.71M  23.6MB/s    in 0.5s    
checkpoints/mit_b1_ 100%[===================>]  50.22M  21.9MB/s    in 2.3s    
Checkpoint sizes:
total 63M
-rw-r--r-- 1 root root 13M Jun 24  2022 mit_b0_imagenet.pth
-rw-r--r-- 1 root root 51M Jun 24  2022 mit_b1_imagenet.pth


In [9]:
# ============================================================
# CELL 5 - Create MMSegmentation Config Files
# ============================================================

import os
os.makedirs('configs', exist_ok=True)

# ── CONFIG 1: SegFormer-B0 Baseline ──────────────────────────────────────────
segformer_b0_config = """
_base_ = [
    'mmseg::_base_/models/segformer_mit-b0.py',
    'mmseg::_base_/default_runtime.py',
    'mmseg::_base_/schedules/schedule_160k.py',
]

# Dataset settings
dataset_type = 'BaseSegDataset'
data_root    = '/content/processed/'
crop_size    = (640, 640)

# Training pipeline
train_pipeline = [
    dict(type='LoadImageFromFile'),
    dict(type='LoadAnnotations'),
    dict(type='RandomResize', scale=(1920, 1080), ratio_range=(0.5, 2.0), keep_ratio=True),
    dict(type='RandomCrop', crop_size=crop_size, cat_max_ratio=0.75),
    dict(type='RandomFlip', prob=0.5),
    dict(type='PhotoMetricDistortion'),
    dict(type='PackSegInputs'),
]

# Validation pipeline
val_pipeline = [
    dict(type='LoadImageFromFile'),
    dict(type='Resize', scale=(1920, 1080), keep_ratio=True),
    dict(type='LoadAnnotations'),
    dict(type='PackSegInputs'),
]

# Dataset meta info (2 classes: background and board)
meta = dict(
    classes=('background', 'board'),
    palette=[[0, 0, 0], [255, 0, 0]]
)

# Dataloaders
train_dataloader = dict(
    batch_size=4, num_workers=2,
    dataset=dict(
        type=dataset_type, data_root=data_root,
        data_prefix=dict(img_path='train/images', seg_map_path='train/masks'),
        img_suffix='.jpeg', seg_map_suffix='.png',
        metainfo=meta, pipeline=train_pipeline,
    )
)

val_dataloader = dict(
    batch_size=1, num_workers=2,
    dataset=dict(
        type=dataset_type, data_root=data_root,
        data_prefix=dict(img_path='val/images', seg_map_path='val/masks'),
        img_suffix='.jpeg', seg_map_suffix='.png',
        metainfo=meta, pipeline=val_pipeline,
    )
)

test_dataloader = val_dataloader

# Evaluation metrics
val_evaluator  = dict(type='IoUMetric', iou_metrics=['mIoU', 'mDice'])
test_evaluator = val_evaluator

# Model: 2 classes
model = dict(
    decode_head=dict(num_classes=2),
    test_cfg=dict(mode='whole')
)

# AdamW optimizer (standard for Transformers)
optim_wrapper = dict(
    type='OptimWrapper',
    optimizer=dict(type='AdamW', lr=6e-5, betas=(0.9, 0.999), weight_decay=0.01),
    paramwise_cfg=dict(
        custom_keys={
            'pos_block': dict(decay_mult=0.),
            'norm':      dict(decay_mult=0.),
            'head':      dict(lr_mult=10.)
        }
    )
)

# LR scheduler with warmup
param_scheduler = [
    dict(type='LinearLR', start_factor=1e-6, by_epoch=False, begin=0,    end=1500),
    dict(type='PolyLR',   eta_min=0.0,       power=1.0,      begin=1500, end=160000, by_epoch=False),
]

# Save best checkpoint based on mIoU
default_hooks = dict(
    checkpoint=dict(type='CheckpointHook', by_epoch=False, interval=2000,
                    max_keep_ckpts=3, save_best='mIoU', rule='greater'),
    logger=dict(type='LoggerHook', interval=50),
)

train_cfg = dict(type='IterBasedTrainLoop', max_iters=160000, val_interval=2000)
val_cfg   = dict(type='ValLoop')
test_cfg  = dict(type='TestLoop')
"""

# ── CONFIG 2: SegFormer-B1 Standard ──────────────────────────────────────────
segformer_b1_config = segformer_b0_config.replace(
    "segformer_mit-b0.py",
    "segformer_mit-b1.py"
)

# Write config files to disk
with open('configs/segformer_b0_baseline.py', 'w') as f:
    f.write(segformer_b0_config)

with open('configs/segformer_b1_standard.py', 'w') as f:
    f.write(segformer_b1_config)

# Verify
print("Config files created:")
!ls -lh configs/

Config files created:
total 8.0K
-rw-r--r-- 1 root root 2.9K Mar 19 16:20 segformer_b0_baseline.py
-rw-r--r-- 1 root root 2.9K Mar 19 16:20 segformer_b1_standard.py


In [11]:
# ============================================================
# CELL 6 - Dataset Verification
# Quick sanity check before launching training
# ============================================================

!pip install ftfy regex -q

from mmengine.config import Config
from mmseg.datasets import BaseSegDataset

# Load config
cfg = Config.fromfile('configs/segformer_b0_baseline.py')

# Test dataset loading
dataset = BaseSegDataset(
    data_root='/content/processed/',
    data_prefix=dict(img_path='train/images', seg_map_path='train/masks'),
    img_suffix='.jpeg',
    seg_map_suffix='.png',
    metainfo=dict(classes=('background', 'board'), palette=[[0,0,0],[255,0,0]])
)

print(f'Train dataset size: {len(dataset)}')
print(f'Expected: 672')

# Check one sample
sample = dataset[0]
print(f'Sample keys: {sample.keys()}')
print('Dataset verification OK!')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.1 MB/s eta 0:00:00


/usr/local/lib/python3.12/dist-packages/mmengine/optim/optimizer/zero_optimizer.py:11: DeprecationWarning: `TorchScript` support for functional optimizers is deprecated and will be removed in a future PyTorch release. Consider using the `torch.compile` optimizer instead.
  from torch.distributed.optim import \


Train dataset size: 672
Expected: 672
Sample keys: dict_keys(['img_path', 'seg_map_path', 'label_map', 'reduce_zero_label', 'seg_fields', 'sample_idx'])
Dataset verification OK!


In [14]:
# ============================================================
# CELL 7 - Push Config Files to GitHub
# Save MMSegmentation config files to the repository
# ============================================================

import os
from getpass import getpass

# Git configuration
!cd /content/repo && git config user.email "massimilianogiangreco7@gmail.com"
!cd /content/repo && git config user.name "ilMassy"

# Ask for token securely (input is hidden)
GITHUB_TOKEN = getpass("Enter your GitHub Personal Access Token: ")

# Set remote URL with authentication token
!cd /content/repo && git remote set-url origin \
    https://{GITHUB_TOKEN}@github.com/ilMassy/advertising-panel-segmentation.git

# Pull remote changes first
!cd /content/repo && git pull origin main --rebase

# Add and commit config files
!cd /content/repo && git add configs/
!cd /content/repo && git commit -m "Add MMSegmentation config files for B0 baseline and B1 standard" \
    || echo "Nothing to commit, already done"

# Push to GitHub
!cd /content/repo && git push origin main

print("Config files pushed to GitHub!")

Enter your GitHub Personal Access Token: ··········
remote: Enumerating objects: 15, done.
remote: Counting objects: 100% (15/15), done.
remote: Compressing objects: 100% (7/7), done.
remote: Total 12 (delta 8), reused 9 (delta 5), pack-reused 0 (from 0)
Unpacking objects: 100% (12/12), 2.43 KiB | 828.00 KiB/s, done.
From https://github.com/ilMassy/advertising-panel-segmentation
 * branch            main       -> FETCH_HEAD
   454efa7..051a763  main       -> origin/main
Successfully rebased and updated refs/heads/main.
On branch main
Your branch is ahead of 'origin/main' by 1 commit.
  (use "git push" to publish your local commits)

nothing to commit, working tree clean
Nothing to commit, already done
Enumerating objects: 6, done.
Counting objects: 100% (6/6), done.
Delta compression using up to 2 threads
Compressing objects: 100% (5/5), done.
Writing objects: 100% (5/5), 1.52 KiB | 1.52 MiB/s, done.
Total 5 (delta 2), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (2

In [ ]:
# ============================================================
# CELL 8 - Training: SegFormer-B0 Baseline (Phase 2)
# Establishes the reference IoU benchmark
# ============================================================

import subprocess, os

# Find mmsegmentation tools directory
import mmseg
mmseg_tools = os.path.join(os.path.dirname(os.path.dirname(mmseg.__file__)), 'tools', 'train.py')
print(f"MMSeg train tool: {mmseg_tools}")

# Launch training
!python {mmseg_tools} \
    configs/segformer_b0_baseline.py \
    --work-dir results/exp0_segformer_b0_baseline \
    --cfg-options \
        model.backbone.init_cfg.checkpoint=checkpoints/mit_b0_imagenet.pth

In [ ]:
# ============================================================
# CELL 9 - Training: SegFormer-B1 Standard (Phase 3)
# Fine-tuning B1 on custom dataset, comparison against B0 baseline
# ============================================================

import os
import mmseg

# Find mmsegmentation tools directory
mmseg_tools = os.path.join(
    os.path.dirname(os.path.dirname(mmseg.__file__)),
    'tools', 'train.py'
)
print(f"MMSeg train tool: {mmseg_tools}")

# Launch B1 standard training
!python {mmseg_tools} \
    configs/segformer_b1_standard.py \
    --work-dir results/exp1_segformer_b1_standard \
    --cfg-options \
        model.backbone.init_cfg.checkpoint=checkpoints/mit_b1_imagenet.pth